In [19]:
import xarray as xr
from pathlib import Path
from itertools import product
from datetime import datetime

KERNEL_FILENAME = "RRTM_kernel_monthly_{year}_alb_TOA_SFC.nc"

def make_kernel_path_alb_cld(year, month):
    return (
        Path(f"{year}") / "monthly_kernel" / "sw" / "alb" / f"RRTM_kernel_cld_alb_TOA_SFC_{month}.nc"
    )

def make_kernel_path_alb_clr(year, month):
    return (
        Path(f"{year}") / "monthly_kernel" / "sw" / "alb" / f"RRTM_kernel_clr_alb_TOA_SFC_{month}.nc"
    )

def make_kernel_path_ts_cld(year, month):
    return (
        Path(f"{year}") / "monthly_kernel" / "lw" / "ts" / f"RRTM_kernel_cld_ts_TOA_SFC_{month}.nc"
    )

def make_kernel_path_ts_clr(year, month):
    return (
        Path(f"{year}") / "monthly_kernel" / "lw" / "ts" / f"RRTM_kernel_clr_ts_TOA_SFC_{month}.nc"
    )

def assign_date_coord(ds: xr.Dataset, year, month):
    return ds.assign_coords(date=datetime(int(year), int(month), 1)).expand_dims(dim="date")

def select_alb_toa_arr(ds: xr.Dataset):
    return ds["TOA"].sel(up_down_net=1, band=1)

def select_ts_toa_arr(ds: xr.Dataset):
    return ds["TOA"].sel(up_down_net=2, band=1)

def build_albedo_kernel_dataset(base, years, months):
    year_months = list(product(years, months))
    kernel_dataarrays_clr = [
        select_alb_toa_arr(assign_date_coord(xr.open_dataset(Path(base) / make_kernel_path_alb_clr(year, month)), year, month))
        for year, month in year_months
    ]
    kernel_dataarrays_cld = [
        select_alb_toa_arr(assign_date_coord(xr.open_dataset(Path(base) / make_kernel_path_alb_cld(year, month)), year, month))
        for year, month in year_months
    ]
    kernel_ds_clr = xr.combine_by_coords(kernel_dataarrays_clr)
    kernel_ds_cld = xr.combine_by_coords(kernel_dataarrays_cld)
    return xr.Dataset(
        data_vars=dict(
            TOA_cld=(["date", "latitude", "longitude"], kernel_ds_cld.TOA.data),
            TOA_clr=(["date", "latitude", "longitude"], kernel_ds_clr.TOA.data),
        ),
        coords=dict(
            latitude=("latitude", kernel_ds_cld.latitude.data),
            longitude=("longitude", kernel_ds_clr.longitude.data),
            date=("date", kernel_ds_clr.date.data),
        ),
    )

def build_ts_kernel_dataset(base, years, months):
    year_months = list(product(years, months))
    cld_arrays = [
        select_ts_toa_arr(assign_date_coord(xr.open_dataset(Path(base) / make_kernel_path_ts_cld(year, month)), year, month))
        for year, month in year_months
    ]
    clr_arrays = [
        select_ts_toa_arr(assign_date_coord(xr.open_dataset(Path(base) / make_kernel_path_ts_clr(year, month)), year, month))
        for year, month in year_months
    ]
    kernel_ds_cld = xr.combine_by_coords(cld_arrays)
    kernel_ds_clr = xr.combine_by_coords(clr_arrays)
    kernel_ds_cld = kernel_ds_cld["TOA"]
    kernel_ds_clr = kernel_ds_clr["TOA"]
    return xr.Dataset(
        data_vars=dict(
            TOA_cld=(["date", "latitude", "longitude"], kernel_ds_cld.data),
            TOA_clr=(["date", "latitude", "longitude"], kernel_ds_clr.data),
        ),
        coords=dict(
            latitude=("latitude", kernel_ds_cld.latitude.data),
            longitude=("longitude", kernel_ds_clr.longitude.data),
            date=("date", kernel_ds_clr.date.data),
        ),
    )

def write_yearly_kernels(ds, output_dir, years):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for year in years:
        path = output_dir / KERNEL_FILENAME.format(year=year)
        kernel_subset = ds.sel(date=(ds.date.dt.year == year))
        kernel_subset.to_netcdf(path)
        print(xr.load_dataset(path))


In [20]:
base = Path("data/kernels_shared")

years = [2011, 2012, 2013, 2014, 2015]
months = [f"{m:02d}" for m in range(1,13)]
print(months, years)

['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12'] [2011, 2012, 2013, 2014, 2015]


In [21]:
fal_kernel_ds = build_albedo_kernel_dataset(base, years, months)
ts_kernel_ds = build_ts_kernel_dataset(base, years, months)

In [22]:
write_yearly_kernels(fal_kernel_ds, "data/fal/kernels", years)
write_yearly_kernels(ts_kernel_ds, "data/ts/kernels", years)

<xarray.Dataset> Size: 1MB
Dimensions:    (date: 12, latitude: 73, longitude: 144)
Coordinates:
  * date       (date) datetime64[ns] 96B 2011-01-01 2011-02-01 ... 2011-12-01
  * latitude   (latitude) float32 292B 90.0 87.5 85.0 82.5 ... -85.0 -87.5 -90.0
  * longitude  (longitude) float32 576B 0.0 2.5 5.0 7.5 ... 352.5 355.0 357.5
Data variables:
    TOA_cld    (date, latitude, longitude) float32 505kB 0.0 0.0 ... -422.3
    TOA_clr    (date, latitude, longitude) float32 505kB 0.0 0.0 ... -455.3
<xarray.Dataset> Size: 1MB
Dimensions:    (date: 12, latitude: 73, longitude: 144)
Coordinates:
  * date       (date) datetime64[ns] 96B 2012-01-01 2012-02-01 ... 2012-12-01
  * latitude   (latitude) float32 292B 90.0 87.5 85.0 82.5 ... -85.0 -87.5 -90.0
  * longitude  (longitude) float32 576B 0.0 2.5 5.0 7.5 ... 352.5 355.0 357.5
Data variables:
    TOA_cld    (date, latitude, longitude) float32 505kB 0.0 0.0 ... -420.6
    TOA_clr    (date, latitude, longitude) float32 505kB 0.0 0.0 ... -453.

In [23]:
for year in years:
    print(xr.load_dataset(f"data/fal/kernels/RRTM_kernel_monthly_{year}_alb_TOA_SFC.nc"))
    print(xr.load_dataset(f"data/ts/kernels/RRTM_kernel_monthly_{year}_alb_TOA_SFC.nc"))


<xarray.Dataset> Size: 1MB
Dimensions:    (date: 12, latitude: 73, longitude: 144)
Coordinates:
  * date       (date) datetime64[ns] 96B 2011-01-01 2011-02-01 ... 2011-12-01
  * latitude   (latitude) float32 292B 90.0 87.5 85.0 82.5 ... -85.0 -87.5 -90.0
  * longitude  (longitude) float32 576B 0.0 2.5 5.0 7.5 ... 352.5 355.0 357.5
Data variables:
    TOA_cld    (date, latitude, longitude) float32 505kB 0.0 0.0 ... -422.3
    TOA_clr    (date, latitude, longitude) float32 505kB 0.0 0.0 ... -455.3
<xarray.Dataset> Size: 1MB
Dimensions:    (date: 12, latitude: 73, longitude: 144)
Coordinates:
  * date       (date) datetime64[ns] 96B 2011-01-01 2011-02-01 ... 2011-12-01
  * latitude   (latitude) float32 292B 90.0 87.5 85.0 82.5 ... -85.0 -87.5 -90.0
  * longitude  (longitude) float32 576B 0.0 2.5 5.0 7.5 ... 352.5 355.0 357.5
Data variables:
    TOA_cld    (date, latitude, longitude) float32 505kB 0.0 0.0 0.0 ... 0.0 0.0
    TOA_clr    (date, latitude, longitude) float32 505kB 0.0 0.0 0.0 